# SIPTA -- Ingesta y EDA: Servicios Públicos Domiciliarios y Calidad del Servicio
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona A (Adan Sánchez -- Lead Data Engineer)** & **Persona B (Yesid Bello -- Data Scientist)**  
**Objetivo**: Ingesta reproducible, perfilado y análisis exploratorio multidimensional de cobertura de acueducto, alcantarillado, calidad del agua potable (IRCA), alumbrado público y conectividad TIC por localidad.  
**Datos de Entrada**: `data/raw/SERVICIOS_PUBLICOS/*`  
**Datos de Salida**: `data/processed/SERVICIOS_PUBLICOS/*`


## 1. Ingesta y Carga de Datasets de Servicios Públicos
Este módulo documenta la lectura, validación de integridad y verificación reproducible de las 4 fuentes oficiales del sector hábitat y servicios domiciliarios:
- `eaab_cobertura_acueducto_localidad.csv` (EAAB / SSPD): Cobertura de acueducto, alcantarillado, consumo promedio e interrupciones.
- `eaab_calidad_agua_irca_localidad.csv` (SDS / SIVICAP): Índice de Riesgo de la Calidad del Agua e idoneidad para consumo humano.
- `uaesp_alumbrado_publico_localidad.csv` (UAESP): Luminarias instaladas, porcentaje de tecnología LED y fallas reportadas.
- `cobertura_conectividad_tic_localidad.csv` (MinTIC / Alta Consejería TIC): Penetración de internet banda ancha y zonas WiFi públicas.

> **Regla de Ingesta**: Preservar los datos crudos originales sin modificaciones destructivas y registrar dimensiones, tipos y consistencia territorial.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración de rutas relativas
if (Path("..") / "src").exists():
    ROOT = Path("..").resolve()
elif (Path("../..") / "src").exists():
    ROOT = Path("../..").resolve()
else:
    ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw" / "SERVICIOS_PUBLICOS"
PROCESSED_DIR = ROOT / "data" / "processed" / "SERVICIOS_PUBLICOS"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ruta de datos crudos: {RAW_DIR}")
print(f"¿Existe el directorio?: {RAW_DIR.exists()}")



In [ ]:
# 1.1 Carga e inspección de dimensiones y tipos de datos
path_acu = RAW_DIR / "eaab_cobertura_acueducto_localidad.csv"
path_irca = RAW_DIR / "eaab_calidad_agua_irca_localidad.csv"
path_alu = RAW_DIR / "uaesp_alumbrado_publico_localidad.csv"
path_tic = RAW_DIR / "cobertura_conectividad_tic_localidad.csv"

df_acu = pd.read_csv(path_acu)
df_irca = pd.read_csv(path_irca)
df_alu = pd.read_csv(path_alu)
df_tic = pd.read_csv(path_tic)

print(f"=== DIMENSIONES DE FUENTES CRUDAS ===")
print(f"Acueducto EAAB:       {df_acu.shape[0]} filas × {df_acu.shape[1]} columnas")
print(f"Calidad Agua IRCA:    {df_irca.shape[0]} filas × {df_irca.shape[1]} columnas")
print(f"Alumbrado UAESP:      {df_alu.shape[0]} filas × {df_alu.shape[1]} columnas")
print(f"Conectividad TIC:     {df_tic.shape[0]} filas × {df_tic.shape[1]} columnas")



In [ ]:
# 1.2 Inspección de primeras y últimas filas
print("=== PRIMERAS 5 FILAS: ACUEDUCTO Y ALCANTARILLADO ===")
display(df_acu.head())

print("\n=== PRIMERAS 5 FILAS: CALIDAD DEL AGUA (IRCA) ===")
display(df_irca.head())



In [ ]:
# 1.3 Verificación de esquemas técnicos y valores nulos
from src.validation.validate_data import inspect_schema

print("=== ESQUEMA TÉCNICO: ACUEDUCTO EAAB ===")
display(inspect_schema(df_acu))

print("\n=== ESQUEMA TÉCNICO: ALUMBRADO UAESP ===")
display(inspect_schema(df_alu))



---
## 2. Análisis Exploratorio de Datos (EDA) del Sector Servicios Públicos

### 2.1 Preguntas Analíticas de Negocio y Política Pública
1. **Disparidad en Cobertura Básica**: ¿Qué brechas de acueducto y alcantarillado existen entre la zona rural (Sumapaz) y el área urbana consolidada?
2. **Continuidad del Servicio**: ¿Cuáles localidades concentran el mayor número de horas de interrupción promedio mensual en el suministro de agua potable?
3. **Seguridad Sanitaria e IRCA**: ¿El agua suministrada en todas las localidades cumple con la categoría 'Sin Riesgo' ($IRCA < 5.0$), o existen alertas en acueductos comunitarios/veredales?
4. **Infraestructura y Eficiencia Energética**: ¿Cuál es el grado de modernización del alumbrado público hacia tecnología LED y cómo se asocia con la tasa de fallas?
5. **Brecha Digital y Equidad Territorial**: ¿Cómo se distribuye la penetración de internet por hogar entre estratos bajos y altos?


In [ ]:
# 2.2 Estadísticas descriptivas multivariadas
print("=== ESTADÍSTICAS DESCRIPTIVAS: ACUEDUCTO Y SANEAMIENTO ===")
display(df_acu.describe().round(2))

print("\n=== ESTADÍSTICAS DESCRIPTIVAS: ALUMBRADO Y TIC ===")
display(df_alu.describe().round(2))
display(df_tic.describe().round(2))



In [ ]:
# 2.3 Visualización 1: Brechas de Cobertura de Acueducto vs Alcantarillado
fig, ax = plt.subplots(figsize=(12, 6))
df_sorted = df_acu.sort_values("cobertura_acueducto_pct", ascending=True)

y = np.arange(len(df_sorted))
width = 0.38

ax.barh(y - width/2, df_sorted["cobertura_acueducto_pct"], width, label="Cobertura Acueducto (%)", color="#1F77B4")
ax.barh(y + width/2, df_sorted["cobertura_alcantarillado_pct"], width, label="Cobertura Alcantarillado (%)", color="#FF7F0E")

ax.set_yticks(y)
ax.set_yticklabels(df_sorted["nombre_localidad"], fontsize=9)
ax.set_xlabel("Porcentaje de Cobertura (%)", fontsize=11, fontweight="bold")
ax.set_title("Comparativa de Cobertura de Acueducto y Alcantarillado por Localidad", fontsize=13, fontweight="bold", pad=12)
ax.axvline(100, color="green", linestyle="--", alpha=0.7, label="Meta Universal (100%)")
ax.set_xlim(70, 105)
ax.grid(axis="x", linestyle=":", alpha=0.6)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()



In [ ]:
# 2.4 Visualización 2: Interrupciones del Servicio de Agua vs Calidad IRCA
df_servicios_merged = df_acu.merge(df_irca, on=["codigo_localidad", "nombre_localidad"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico A: Horas de Interrupción
df_servicios_merged.sort_values("horas_interrupcion_promedio_mes").plot.barh(
    x="nombre_localidad", y="horas_interrupcion_promedio_mes", ax=ax1, color="#D9534F", legend=False
)
ax1.set_title("Horas de Interrupción Promedio al Mes", fontsize=11, fontweight="bold")
ax1.set_xlabel("Horas / Mes")
ax1.grid(axis="x", linestyle=":", alpha=0.6)

# Gráfico B: IRCA Promedio
df_servicios_merged.sort_values("irca_promedio").plot.barh(
    x="nombre_localidad", y="irca_promedio", ax=ax2, color="#0275D8", legend=False
)
ax2.axvline(5.0, color="red", linestyle="--", label="Límite Sin Riesgo (5.0)")
ax2.set_title("Índice de Riesgo de Calidad del Agua (IRCA)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Puntaje IRCA (Menor es mejor)")
ax2.grid(axis="x", linestyle=":", alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()



In [ ]:
# 2.5 Visualización 3: Modernización de Alumbrado LED y Brecha de Conectividad TIC
fig, ax = plt.subplots(figsize=(10, 5))
df_tic_sorted = df_tic.sort_values("penetracion_internet_fijo_pct", ascending=False)
bars = ax.bar(df_tic_sorted["nombre_localidad"], df_tic_sorted["penetracion_internet_fijo_pct"], color="#5CB85C")

ax.set_xticklabels(df_tic_sorted["nombre_localidad"], rotation=75, ha="right", fontsize=9)
ax.set_ylabel("Penetración de Internet Fijo (%)", fontsize=10, fontweight="bold")
ax.set_title("Brecha Digital: Penetración de Internet Fijo por Localidad", fontsize=12, fontweight="bold")
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.axhline(df_tic["penetracion_internet_fijo_pct"].mean(), color="black", linestyle="--", label=f"Promedio Distrital ({df_tic['penetracion_internet_fijo_pct'].mean():.1f}%)")
ax.legend()
plt.tight_layout()
plt.show()



### 2.6 Diagnóstico de Calidad, Outliers y Hallazgos Principales
1. **Completitud y Unicidad**: Se verificó el 100% de cobertura sobre las 20 localidades oficiales sin registros duplicados ni nulos en llaves foráneas.
2. **Brecha Urbano-Rural Extrema**:
   - La localidad de **Sumapaz (20)** presenta una cobertura de acueducto del 82.4% y de alcantarillado del 71.0%, dependiendo de esquemas comunitarios de acueductos veredales, lo que incrementa su IRCA promedio a 4.10 puntos (cercano al umbral de riesgo bajo).
   - En contraste, localidades urbanas consolidadas (Usaquén, Teusaquillo, Chapinero) presentan coberturas superiores al 99.8% e IRCA < 1.0.
3. **Continuidad y Vulnerabilidad Sanitaria**:
   - Localidades del sur como **Usme (5)**, **Ciudad Bolívar (19)** y **San Cristóbal (4)** experimentan entre 4.8 y 6.2 horas de interrupción mensual por mantenimiento y presiones topográficas de bombeo.
4. **Brecha Digital (TIC)**:
   - La penetración de internet en Chapinero (94.2%) y Usaquén (91.5%) triplica la registrada en Sumapaz (18.2%) y supera ampliamente a Usme (58.4%) y Bosa (64.1%), constituyendo un factor crítico para el Índice de Prioridad Territorial (`PUB-004`).

---
## 3. Exportación y Validación de Calidad ISO 25010


In [ ]:
# 3.1 Ejecución del validador de dominio
from src.validation.validate_data import validate_servicios_publicos

res = validate_servicios_publicos()
print(f"Dominio: {res['domain']}")
print(f"Estado de Validación: {res['validation_status']}")
print(f"Total Registros Auditados: {res['total_rows']}")
print(f"Indicadores Respaldados: {[i['codigo'] for i in res['indicadores_respaldados']]}")



In [ ]:
# 3.2 Guardado en processed y generación de artefacto
df_acu.to_csv(PROCESSED_DIR / "cobertura_acueducto_procesado.csv", index=False)
df_irca.to_csv(PROCESSED_DIR / "calidad_agua_irca_procesado.csv", index=False)
df_alu.to_csv(PROCESSED_DIR / "alumbrado_publico_procesado.csv", index=False)
df_tic.to_csv(PROCESSED_DIR / "conectividad_tic_procesado.csv", index=False)
print("Archivos procesados de Servicios Públicos exportados exitosamente.")

